In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os

def crawl_saramin():
    base_url = "https://www.saramin.co.kr/zf_user/search"
    params = {
        'search_area': 'main',
        'search_done': 'y', 
        'search_optional_item': 'n',
        'searchType': 'search',
        'searchword': '데이터분석'
    }
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    response = requests.get(base_url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    jobs = []
    job_items = soup.find_all('div', class_='item_recruit')
    
    for item in job_items:

        company_elem = item.find('strong', class_='corp_name')
        company = company_elem.get_text(strip=True) if company_elem else 'N/A'

        title_elem = item.find('h2', class_='job_tit')
        title = 'N/A'
        url = 'N/A'
        
        if title_elem:
            title_link = title_elem.find('a')
            if title_link:
                title = title_link.get_text(strip=True)
                href = title_link.get('href', '')
                if href.startswith('/zf_user'):
                    url = f"https://www.saramin.co.kr{href}"
                elif href.startswith('http'):
                    url = href
        
        condition_elem = item.find('div', class_='job_condition')
        conditions = []
        if condition_elem:
            condition_spans = condition_elem.find_all('span')
            for span in condition_spans:
                text = span.get_text(strip=True)
                if text and text not in ['↑', '↓', '|']:
                    conditions.append(text)
        
        requirement = ' | '.join(conditions) if conditions else 'N/A'
        
        jobs.append({
            'Site': 'Saramin',
            'Col_Company': company,
            'Col_Recruit': title,
            'Col_detail': requirement,
            'Col_URL': url
            
        })
    
    return pd.DataFrame(jobs)

df_saramin = crawl_saramin()

if not os.path.exists('data_tmp'):
    os.makedirs('data_tmp')

df_saramin.to_csv('data_tmp/data_saramin.csv', index=False, encoding='utf-8-sig')


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os
from urllib.parse import urljoin

def crawl_jobkorea_data():
    
    # 검색 URL
    url = "https://www.jobkorea.co.kr/Search/?stext=%EB%8D%B0%EC%9D%B4%ED%84%B0%EB%B6%84%EC%84%9D&Page_No=1"
    
    # 헤더 설정 (실제 브라우저처럼 보이게)
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ko-KR,ko;q=0.9,en;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
    }
    
 
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    
    soup = BeautifulSoup(response.content, 'html.parser')
    job_data = []

    job_list = []
    
    title_links = soup.find_all('a', href=lambda href: href and '/Recruit/GI_Read/' in href)
    print(f"제목 링크 수: {len(title_links)}")
    
    processed_urls = set() 
    
    for link in title_links:
        href = link.get('href', '')
        if href in processed_urls:
            continue
        processed_urls.add(href)
        
        current = link
        job_container = None
        
        for level in range(7): 
            current = current.parent
            if current is None:
                break
            
            if current.name in ['div', 'li', 'article']:
                company_check = current.find('span', class_=lambda x: x and 'Typography_variant_size16' in str(x))
                detail_check = current.find('div', class_=lambda x: x and 'Flex_gap_space16' in str(x))
                
                if company_check and detail_check:
                    job_container = current
                    break
                elif level >= 4: 
                    job_container = current
                    break
        
        if job_container:
            job_list.append(job_container)
    
    print(f"찾은 채용공고 컨테이너 수: {len(job_list)}")
    
    for idx, job_container in enumerate(job_list, 1):
        company_elem = job_container.find('a', style=lambda style: style and 'max-width:120px' in style)
        if not company_elem:
            company_elem = job_container.find('span', class_=lambda x: x and 'Typography_variant_size16' in str(x))
        
        company = company_elem.get_text(strip=True) if company_elem else "회사명 없음"
        
        title_elem = job_container.find('a', class_='h7nnv12')
        if title_elem:
            title_span = title_elem.find('span', class_=lambda x: x and 'Typography_variant_size18' in str(x) and 'Typography_truncate' in str(x))
            if title_span:
                title = title_span.get_text(strip=True)
            else:
                title = title_elem.get_text(strip=True)
            
            job_url = urljoin("https://www.jobkorea.co.kr", title_elem['href'])
        else:
            title = "제목 없음"
            job_url = "URL 없음"
        
        detail_div = job_container.find('div', class_=lambda x: x and 'Flex_gap_space16' in str(x))
        if detail_div:
            detail_spans = detail_div.find_all('span', class_=lambda x: x and 'Typography_variant_size14' in str(x) and 'Typography_color_gray800' in str(x))
            details = [span.get_text(strip=True) for span in detail_spans if span.get_text(strip=True)]
            detail = ' | '.join(details) if details else "상세정보 없음"
        else:
            all_spans = job_container.find_all('span')
            details = []
            for span in all_spans:
                text = span.get_text(strip=True)
                if (text and text not in [title, company] and 
                    any(keyword in text for keyword in ['경력', '학력', '정규직', '계약직', '인턴', '구', '시', '월', '일'])):
                    details.append(text)
            detail = ' | '.join(details[:5]) if details else "상세정보 없음"  # 최대 5개만
        
        job_data.append({
            'Site': 'Job_Korea',
            'Col_company': company,
            'Col_Recruit': title,
            'Col_detail': detail,
            'Col_url': job_url
        })
        
        print(f"{idx}. {company} - {title}")
            
        
    

    df = pd.DataFrame(job_data)
    return df
    
            


def save_to_csv(df, folder="data_tmp", filename="jobkorea_data_analysis.csv"):

    if not df.empty:
        if not os.path.exists(folder):
            os.makedirs(folder)
            print(f"'{folder}' 폴더를 생성했습니다.")
        
        filepath = os.path.join(folder, filename)
    
        df.to_csv(filepath, index=False, encoding='utf-8-sig')
        print(f"데이터가 '{filepath}' 파일로 저장되었습니다.")
    else:
        print("저장할 데이터가 없습니다.")

def main():
    
    df = crawl_jobkorea_data()
    
if __name__ == "__main__":
    main()